# Demonstração de Diagnóstico de Diabetes com IA
Este notebook demonstra todo o pipeline programaticamente: pré-processamento, inferência de Machine Learning e explicação com **LLama via Ollama**.

In [ ]:
import pandas as pd
import numpy as np
import joblib
import sys
sys.path.append('..') # Para importar da raiz do projeto
from llm_interpreter import interpretar_resultado

### 1. Carregando Pipeline do Modelo

In [ ]:
# Certifique-se de que os modelos já foram treinados (`python models/train_model.py`)
modelo = joblib.load('../model_diabetes.pkl')
imputer = joblib.load('../imputer.pkl')
scaler = joblib.load('../scaler.pkl')

print("Modelos e pré-processadores carregados!")

### 2. Definindo Paciente de Exemplo

In [ ]:
dados_paciente = {
    'Pregnancies': 3,
    'Glucose': 125,
    'BloodPressure': 76,
    'SkinThickness': 25,
    'Insulin': 90,
    'BMI': 28.5,
    'DiabetesPedigreeFunction': 0.65,
    'Age': 42
}

df_paciente = pd.DataFrame([dados_paciente])
df_paciente

### 3. Pré-Processamento idêntico ao Treinamento

In [ ]:
colunas_erro = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
df_paciente[colunas_erro] = df_paciente[colunas_erro].replace(0, np.nan)

df_imputed_array = imputer.transform(df_paciente)
df_imputed = pd.DataFrame(df_imputed_array, columns=df_paciente.columns)
df_imputed['Glucose_Insulin'] = df_imputed['Glucose'] * df_imputed['Insulin']

X_final_array = scaler.transform(df_imputed)
X_final = pd.DataFrame(X_final_array, columns=df_imputed.columns)

print("Dados formatados:")
X_final

### 4. Predição com o Modelo de ML

In [ ]:
predicao = modelo.predict(X_final)[0]
probabilidades = modelo.predict_proba(X_final)[0]

confianca = probabilidades[1] if predicao == 1 else probabilidades[0]
resultado_str = "POSITIVO" if predicao == 1 else "NEGATIVO"

print(f"Diagnóstico Final: {resultado_str}")
print(f"Confiança da IA: {confianca:.1%}")

### 5. Interpretação em Linguagem Natural (LLama via Ollama)

In [ ]:
from IPython.display import display, Markdown

# IMPORTANTE: O Ollama precisa estar rodando na máquina (`ollama serve`)
print("Aguardando análise da LLM...\n")
texto_llm = interpretar_resultado(dados_paciente, predicao, confianca)

if texto_llm:
    display(Markdown(texto_llm))
else:
    print("Não foi possível obter a resposta. Verifique o Ollama.")